| Method | On-Policy | Off-Policy |
|---|---|---|
| **Value-Based** | SARSA | Q-Learning, DQN |
| **Actor-Critic** | A2C, PPO | TD3, SAC |
| **Pure Policy Gradient** | REINFORCE | Off-Policy REINFORCE (with Importance Sampling Correction) |

## Why Do We Want Multiple Gradient Updates on the Same Batch of Data?

Suppose REINFORCE collects **1,000 transitions**, performs only **one gradient update**, and then discards the data.

Why would we want to perform multiple gradient updates on the same batch of data? And why does doing so introduce new problems?

### REINFORCE: The Original Approach

1. Collect data using the current policy.
2. Perform one gradient update.
3. Collect new data using the updated policy.
4. Perform another gradient update.
5. Repeat.

This is inefficient because **environment interaction is expensive!**

Instead, we would prefer to:

1. Collect a large batch of data using the current policy.
2. Perform multiple gradient updates using the same batch.
3. Collect new data only after several updates.

### The Fundamental Trade-off

There are two competing objectives:

1. **Sample Efficiency:** Interacting with the environment is expensive, so we want to reuse the same batch of data for multiple training updates.

2. **Gradient Correctness:** After each policy update, the previously collected data no longer comes from the current policy. If we directly reuse that data to compute the REINFORCE gradient, the resulting gradient estimate may become **biased**.

**The core challenge:** How can we reuse collected experience to improve sample efficiency without introducing excessive bias into our policy gradient estimates?

In [1]:
import numpy as np

rng = np.random.default_rng(42)

old = np.array([0.8, 0.2])
new = np.array([0.5, 0.5])
rewards = np.array([0.0, 10.0])

actions = rng.choice(2, size=10_000, p=old)

ratio = new[actions] / old[actions]

print("Old batch average:", rewards[actions].mean())
print("IS estimate:", np.mean(ratio * rewards[actions]))
print("True new expectation:", new @ rewards)

Old batch average: 1.954
IS estimate: 4.885
True new expectation: 5.0


### Importance Sampling (IS)

Here comes the magic:

**We haven't sampled a single action from the New Policy, yet using only data collected by the Old Policy, we can estimate the New Policy's Expected Reward with high accuracy!**

This is **Importance Sampling (IS)**.

Imagine the Old Policy collected 1,000 actions.

If we directly use the Old Policy's data, A would be overrepresented, while B would be underrepresented.

To correct this, we assign a **weight** to each sample:

$$
w(a)=\frac{\pi_{\text{new}}(a)}{\pi_{\text{old}}(a)}
$$

For A:

$$
w(A)=\frac{0.5}{0.8}=0.625
$$

For B:

$$
w(B)=\frac{0.5}{0.2}=2.5
$$

Therefore:

- The Old Batch contains approximately 800 samples of A. Multiplying each by 0.625 gives an effective weight equivalent to 500 samples of A.
- The Old Batch contains approximately 200 samples of B. Multiplying each by 2.5 gives an effective weight equivalent to 500 samples of B.

**We haven't collected any new samples. Instead, through reweighting, we've made the old samples represent the new probability distribution in expectation.**

This is the intuition behind Importance Sampling.

During a single batch of training:

- **Old Policy** remains fixed.
- **Rewards and Advantage Estimates** remain fixed.
- **New Policy** parameters are continuously updated through gradient updates.
- **Importance Ratios** change as the New Policy changes.
- **Clipping** limits the incentive to keep pushing action probabilities too far in the same direction.

---

Suppose an action has an **Advantage of +5**. The Old Policy assigned action A a probability of 50%, but the New Policy has already increased that probability to 70%.

Since A is a good action, **why does PPO prefer to stop extracting additional optimization gains from this old sample, rather than continuing to encourage the policy to push A's probability toward 90%?**